# RAG System for Plant Disease Research

## 1. Setup and Imports

In [1]:
# Install firebase if not already installed
!pip install firebase PyPDF2 nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.5 MB/s eta 0:00:00


In [33]:
import re
import google.generativeai as genai
from firebase.firebase import FirebaseApplication
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer
import datetime # Import datetime module for local timestamps
import PyPDF2 # New import for PDF handling
import tempfile # New import for temporary file handling
import os # New import for file operations

# Initialize Firebase with your database URL
# The 'None' here means no authentication token is provided.
# This setup requires your Firebase Realtime Database security rules to allow unauthenticated write access.
# If you encounter a 401 error, you need to adjust your database rules in the Firebase console.
FBconn = FirebaseApplication('https://basil-plant-disease-default-rtdb.firebaseio.com/', None)

# Configure your API key for the Gemini API
# This key was previously extracted from the kernel state.
API_KEY = ''
genai.configure(api_key=API_KEY)

## 2. RAG System Class Definition

In [3]:
class RAGSystem:
    def __init__(self):
        print("Initializing RAG System...")
        self.doc_id_mapping = {
            1: "https://doi.org/10.3390/plants9050654",
            2: "https://doi.org/10.1094/pdis.1997.81.2.124",
            3: "https://doi.org/10.21273/hortsci09778-16",
            4: "https://doi.org/10.3732/apps.1300032",
            5: "https://doi.org/10.21273/horttech03849-17"
        }

        from nltk.stem import PorterStemmer # Import here to ensure it's available within the class
        self.stemmer = PorterStemmer()

        original_inverted_index = {
            "basil": [1, 2, 3, 5], "disease": [1, 2, 3, 4, 5], "downy": [1, 5],
            "mildew": [1, 5], "management": [1, 2, 3, 5], "infection": [1, 3, 5],
            "resistance": [1, 2, 3, 5], "peronospora": [5], "belbahrii": [5],
            "sporulation": [3, 5], "chlorosis": [5], "oomycete": [5], "spore": [5],
            "greenhouse": [2, 3, 5], "inoculum": [3, 5], "epidemiology": [5],
            "fusarium": [2], "wilt": [2], "mould": [3], "white": [3],
            "epidermis": [5], "temperature": [3, 5]
        }
        self.inverted_index = {}
        for term, doc_ids in original_inverted_index.items():
            self.inverted_index[self.stemmer.stem(term)] = doc_ids

        print("\n--- RAGSystem Stemmed Inverted Index (sample) ---")
        # Print a sample to avoid excessive output, or full for debugging
        sample_keys = list(self.inverted_index.keys())[:5]
        sample_index = {k: self.inverted_index[k] for k in sample_keys}
        print(sample_index)
        print("--------------------------------------------------")

        self.stop_words = {
            "the", "and", "of", "in", "to", "a", "is", "for", "on", "with", "by",
            "from", "at", "as", "it", "he", "she", "they", "we", "you", "that", "this",
            "but", "or", "not", "has", "have", "had", "do", "does", "did", "can",
            "will", "would", "should", "could", "may", "might", "must"
        }

        # Mock corpus to simulate retrieved documents for the RAG pipeline
        self.mock_corpus = {
            1: "Downy mildew is a severe disease in basil. Management requires tracking infection rates and breeding for resistance.",
            2: "Fusarium wilt causes severe disease in greenhouse basil. Proper greenhouse management and disease resistance are key.",
            3: "White mould infection in greenhouse basil is affected by temperature. Managing inoculum and sporulation helps build resistance.",
            4: "General disease management strategies in plants.",
            5: "The oomycete Peronospora belbahrii causes downy mildew in basil. Epidemiology shows temperature impacts spore inoculum and sporulation. Symptoms include chlorosis on the epidermis. Effective disease management and resistance are critical in the greenhouse."
        }
        # Initialize Gemini Model
        self.gemini_model = genai.GenerativeModel('gemini-pro-latest')


    def retrieve_documents(self, query_keywords, top_n=2):
        relevant_doc_ids = set()
        for keyword in query_keywords:
            stemmed_keyword = self.stemmer.stem(keyword.lower())
            if stemmed_keyword in self.inverted_index:
                for doc_id in self.inverted_index[stemmed_keyword]:
                    relevant_doc_ids.add(doc_id)

        sorted_relevant_docs = sorted(list(relevant_doc_ids))
        top_documents = []

        for doc_id in sorted_relevant_docs[:top_n]:
            if doc_id in self.mock_corpus:
                top_documents.append({'id': doc_id, 'text': self.mock_corpus[doc_id]})
        return top_documents

    def generate_rag_response(self, question, context):
        if not context:
            return "i could not find an answer please ask something else"

        context_text = "\n".join([f"Source [{doc['id']}] ({self.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in context])

        prompt = f"""Given the following context, answer the question comprehensively.
        Context:
        {context_text}

        Question: {question}
        Answer:"""

        print("\n--- RAG Prompt (Ready to be sent to LLM) ---")
        print(prompt)
        print("------------------------------------------")

        # Actual LLM integration
        try:
            response = self.gemini_model.generate_content(prompt)
            generated_answer = response.text.strip()
            if not generated_answer:
                return "i could not find an answer to this question according the articles"
            return generated_answer
        except Exception as e:
            return "i could not find an answer please ask something else"

    def get_existing_queries(self, fb_connection):
        try:
            all_rag_results = fb_connection.get('/rag_search_results/', None)
            if all_rag_results:
                return {value.get('query').lower(): key for key, value in all_rag_results.items() if value.get('query')}
            return {}
        except Exception as e:
            print(f"Error fetching existing queries from Firebase: {e}")
            return {}

## 3. RAG Configuration & Knowledge Base Management (Firebase)

In [4]:
# Assuming 'rag_instance' is already initialized from the previous cell

try:
    # Store doc_id_mapping
    FBconn.put('/rag_config/', 'doc_id_mapping', rag_instance.doc_id_mapping)
    print("Successfully posted doc_id_mapping to Firebase.")

    # Store inverted_index (could be large, consider breaking it down if needed)
    FBconn.put('/rag_config/', 'inverted_index', rag_instance.inverted_index)
    print("Successfully posted inverted_index to Firebase.")

    # Store mock_corpus
    FBconn.put('/rag_config/', 'mock_corpus', rag_instance.mock_corpus)
    print("Successfully posted mock_corpus to Firebase.")

    # Optionally, store a timestamp for this configuration update
    FBconn.post('/rag_config_updates/', {'timestamp': datetime.datetime.now().isoformat(), 'message': 'RAG system configuration updated'})
    print("Configuration update timestamp posted to Firebase.")

except Exception as e:
    print(f"Error posting RAG configuration to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_config/' and '/rag_config_updates/' to ensure write access is permitted.")

Error posting RAG configuration to Firebase: name 'rag_instance' is not defined
Please check your Firebase database rules for '/rag_config/' and '/rag_config_updates/' to ensure write access is permitted.


In [5]:
# Assuming 'rag_instance' is already initialized and updated from the previous cells

try:
    # Store doc_id_mapping
    FBconn.put('/rag_config/', 'doc_id_mapping', rag_instance.doc_id_mapping)
    print("Successfully posted doc_id_mapping to Firebase.")

    # Store inverted_index (could be large, consider breaking it down if needed)
    FBconn.put('/rag_config/', 'inverted_index', rag_instance.inverted_index)
    print("Successfully posted inverted_index to Firebase.")

    # Store mock_corpus
    FBconn.put('/rag_config/', 'mock_corpus', rag_instance.mock_corpus)
    print("Successfully posted mock_corpus to Firebase.")

    # Optionally, store a timestamp for this configuration update
    FBconn.post('/rag_config_updates/', {'timestamp': datetime.datetime.now().isoformat(), 'message': 'RAG system configuration re-updated'})
    print("Configuration re-update timestamp posted to Firebase.")

except Exception as e:
    print(f"Error posting RAG configuration to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_config/' and '/rag_config_updates/' to ensure write access is permitted.")

Error posting RAG configuration to Firebase: name 'rag_instance' is not defined
Please check your Firebase database rules for '/rag_config/' and '/rag_config_updates/' to ensure write access is permitted.


## 4. RAG Pipeline Execution and Firebase Updates

In [6]:
rag_instance = RAGSystem()

# Fetch existing queries from Firebase
existing_queries = rag_instance.get_existing_queries(FBconn)
print(f"Found {len(existing_queries)} existing queries in Firebase.")

# The LLM is now enabled. No need to configure the API key here again.

# Test Query 1 - Re-processing to find an answer based on articles
test_query = "What causes downy mildew in basil and what are the symptoms?"
print(f"\nRe-processing Query 1: {test_query}")

# Explicitly construct context from relevant documents in mock_corpus
# Documents 1 and 5 directly mention downy mildew, basil, and symptoms
relevant_docs_for_mildew = [
    {'id': 1, 'text': rag_instance.mock_corpus[1]},
    {'id': 5, 'text': rag_instance.mock_corpus[5]}
]

# Ensure retrieved_context uses these specific documents
retrieved_context = relevant_docs_for_mildew
context_q1 = "\n".join([f"Source [{doc['id']}] ({rag_instance.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in retrieved_context])

print(f"Explicitly constructed Context for Query 1:\n{context_q1}")

# This will now use the actual LLM to generate a new answer
answer = rag_instance.generate_rag_response(test_query, retrieved_context)
print(f"Generated Answer 1: {answer}")

# Get the existing entry ID for this query from the previous output
existing_entry_id_q1 = '-OtnPjTBxp8IbGIcQpgs' # This ID was confirmed in the `remaining_results` output

# Update the answer and context for this specific entry in Firebase
try:
    rag_data_update = {
        'answer': answer,
        'context': context_q1,
        'timestamp_updated': datetime.datetime.now().isoformat() # Add an update timestamp
    }
    FBconn.patch(f'/rag_search_results/{existing_entry_id_q1}/', rag_data_update)
    print(f"Successfully updated Query 1 data for ID {existing_entry_id_q1} in Firebase with new answer.")
except Exception as e:
    print(f"Error updating Query 1 data for ID {existing_entry_id_q1} in Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")

# Test Query 2 - Remains as is (already processed with an answer)
test_query2 = "How does temperature affect greenhouse diseases?"
print(f"\nQuery 2: {test_query2} (Already processed in previous turn)")
existing_entry_id_q2 = '-OtnPjU6U3YvVEXWCCCG'
existing_entry_q2 = FBconn.get(f'/rag_search_results/{existing_entry_id_q2}', None)
if existing_entry_q2:
    print(f"Existing Answer: {existing_entry_q2.get('answer', 'N/A')}")
else:
    print("Could not retrieve existing answer for Query 2.")

Initializing RAG System...

--- RAGSystem Stemmed Inverted Index (sample) ---
{'basil': [1, 2, 3, 5], 'diseas': [1, 2, 3, 4, 5], 'downi': [1, 5], 'mildew': [1, 5], 'manag': [1, 2, 3, 5]}
--------------------------------------------------
Found 4 existing queries in Firebase.

Re-processing Query 1: What causes downy mildew in basil and what are the symptoms?
Explicitly constructed Context for Query 1:
Source [1] (https://doi.org/10.3390/plants9050654): Downy mildew is a severe disease in basil. Management requires tracking infection rates and breeding for resistance.
Source [5] (https://doi.org/10.21273/horttech03849-17): The oomycete Peronospora belbahrii causes downy mildew in basil. Epidemiology shows temperature impacts spore inoculum and sporulation. Symptoms include chlorosis on the epidermis. Effective disease management and resistance are critical in the greenhouse.

--- RAG Prompt (Ready to be sent to LLM) ---
Given the following context, answer the question comprehensively.
 

In [7]:
specific_entry_id = '-OtnPjTBxp8IbGIcQpgs'
new_context_derived_answer = "Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5]."

try:
    # Update only the 'answer' field of the specific entry
    FBconn.patch(f'/rag_search_results/{specific_entry_id}/', {'answer': new_context_derived_answer, 'timestamp_updated': datetime.datetime.now().isoformat()})
    print(f"Successfully updated answer for entry ID {specific_entry_id} in Firebase with context-derived answer.")

except Exception as e:
    print(f"Error updating entry ID {specific_entry_id} in Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")

Successfully updated answer for entry ID -OtnPjTBxp8IbGIcQpgs in Firebase with context-derived answer.


In [8]:
specific_entry_id = '-OtnPjTBxp8IbGIcQpgs'

try:
    # Get the specific RAG search result
    updated_entry = FBconn.get(f'/rag_search_results/{specific_entry_id}', None)

    if updated_entry:
        print(f"Successfully retrieved updated RAG search result for ID: {specific_entry_id}")
        print("\n--- Updated RAG Search Result ---")
        print(f"  Query: {updated_entry.get('query', 'N/A')}")
        print(f"  Answer: {updated_entry.get('answer', 'N/A')}")
        print(f"  Context: {updated_entry.get('context', 'N/A')[:200]}...") # Truncate long context
        print(f"  Timestamp: {updated_entry.get('timestamp', 'N/A')}")
        print(f"  Timestamp Updated: {updated_entry.get('timestamp_updated', 'N/A')}")
        print("--------------------------------------------------")
    else:
        print(f"No RAG search result found for ID: {specific_entry_id}.")

except Exception as e:
    print(f"Error retrieving RAG search result for ID {specific_entry_id} from Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure read access is permitted.")

Successfully retrieved updated RAG search result for ID: -OtnPjTBxp8IbGIcQpgs

--- Updated RAG Search Result ---
  Query: What causes downy mildew in basil and what are the symptoms?
  Answer: Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5].
  Context: Source [1] (https://doi.org/10.3390/plants9050654): Downy mildew is a severe disease in basil. Management requires tracking infection rates and breeding for resistance.
Source [5] (https://doi.org/10....
  Timestamp: 2026-05-29T11:07:30.525740
  Timestamp Updated: 2026-05-31T05:52:07.098660
--------------------------------------------------


In [9]:
# --- New Query 1: What causes fusarium wilt in basil? ---
print("\n--- Processing New Query 1 ---")
new_query_1 = "What causes fusarium wilt in basil?"

# Manually select relevant documents for this query from mock_corpus
# Article 2 is directly relevant
relevant_docs_q1 = [
    {'id': 2, 'text': rag_instance.mock_corpus[2]}
]
context_new_q1 = "\n".join([f"Source [{doc['id']}] ({rag_instance.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in relevant_docs_q1])

print(f"New Query 1: {new_query_1}")
print(f"Context for New Query 1:\n{context_new_q1}")

answer_new_q1 = rag_instance.generate_rag_response(new_query_1, relevant_docs_q1)
print(f"Generated Answer for New Query 1: {answer_new_q1}")

# Post the new query and answer to Firebase
try:
    rag_data_new_q1 = {
        'query': new_query_1,
        'answer': answer_new_q1,
        'context': context_new_q1,
        'timestamp': datetime.datetime.now().isoformat()
    }
    post_result_q1 = FBconn.post('/rag_search_results/', rag_data_new_q1)
    print(f"Successfully posted New Query 1 to Firebase with ID: {post_result_q1['name']}")
except Exception as e:
    print(f"Error posting New Query 1 to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")


# --- New Query 2: What affects white mould infection in greenhouse basil? ---
print("\n--- Processing New Query 2 ---")
new_query_2 = "What affects white mould infection in greenhouse basil?"

# Manually select relevant documents for this query from mock_corpus
# Article 3 is directly relevant
relevant_docs_q2 = [
    {'id': 3, 'text': rag_instance.mock_corpus[3]}
]
context_new_q2 = "\n".join([f"Source [{doc['id']}] ({rag_instance.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in relevant_docs_q2])

print(f"New Query 2: {new_query_2}")
print(f"Context for New Query 2:\n{context_new_q2}")

answer_new_q2 = rag_instance.generate_rag_response(new_query_2, relevant_docs_q2)
print(f"Generated Answer for New Query 2: {answer_new_q2}")

# Post the new query and answer to Firebase
try:
    rag_data_new_q2 = {
        'query': new_query_2,
        'answer': answer_new_q2,
        'context': context_new_q2,
        'timestamp': datetime.datetime.now().isoformat()
    }
    post_result_q2 = FBconn.post('/rag_search_results/', rag_data_new_q2)
    print(f"Successfully posted New Query 2 to Firebase with ID: {post_result_q2['name']}")
except Exception as e:
    print(f"Error posting New Query 2 to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")


--- Processing New Query 1 ---
New Query 1: What causes fusarium wilt in basil?
Context for New Query 1:
Source [2] (https://doi.org/10.1094/pdis.1997.81.2.124): Fusarium wilt causes severe disease in greenhouse basil. Proper greenhouse management and disease resistance are key.

--- RAG Prompt (Ready to be sent to LLM) ---
Given the following context, answer the question comprehensively.
        Context:
        Source [2] (https://doi.org/10.1094/pdis.1997.81.2.124): Fusarium wilt causes severe disease in greenhouse basil. Proper greenhouse management and disease resistance are key.

        Question: What causes fusarium wilt in basil?
        Answer:
------------------------------------------
Generated Answer for New Query 1: Based on the provided context, the specific cause of Fusarium wilt is not mentioned. Instead, the text states that Fusarium wilt *is* the cause of a severe disease in greenhouse basil. 

To manage and address this disease, the context notes that proper greenh

In [10]:
print("Retrieving all RAG entries from Firebase to confirm additions...")
all_current_results = FBconn.get('/rag_search_results/', None)

if all_current_results:
    print(f"Found {len(all_current_results)} total entries.")
    for entry_id, entry_data in all_current_results.items():
        query = entry_data.get('query', 'N/A')
        answer = entry_data.get('answer', 'N/A')
        timestamp = entry_data.get('timestamp', 'N/A')
        timestamp_updated = entry_data.get('timestamp_updated', 'N/A')
        print(f"\n--- Entry ID: {entry_id} ---")
        print(f"  Query: {query}")
        print(f"  Answer: {answer}")
        print(f"  Timestamp: {timestamp}")
        if timestamp_updated != 'N/A':
            print(f"  Timestamp Updated: {timestamp_updated}")
else:
    print("No RAG entries found in Firebase.")

print("Display of all current entries finished.")

Retrieving all RAG entries from Firebase to confirm additions...
Found 6 total entries.

--- Entry ID: -OtnPjTBxp8IbGIcQpgs ---
  Query: What causes downy mildew in basil and what are the symptoms?
  Answer: Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5].
  Timestamp: 2026-05-29T11:07:30.525740
  Timestamp Updated: 2026-05-31T05:52:07.098660

--- Entry ID: -OtnPjU6U3YvVEXWCCCG ---
  Query: How does temperature affect greenhouse diseases?
  Answer: Based on the provided context, temperature significantly affects greenhouse diseases, specifically in basil crops, in the following ways:

*   **Influences Infection:** Temperature directly affects the infection rate and development of white mould [3]. 
*   **Impacts Spore Inoculum and Sporulation:** For diseases like downy mildew (caused by the oomycete *Peronospora belbahrii

## 5. Firebase Data Cleaning

In [11]:
# Fetch all RAG search results
print("Fetching all RAG search results from Firebase...")
all_rag_results = FBconn.get('/rag_search_results/', None)

if not all_rag_results:
    print("No RAG search results found in Firebase.")
else:
    print(f"Found {len(all_rag_results)} total entries.")

    # Dictionary to store unique queries and their first entry ID
    unique_queries = {}
    # List to store IDs of duplicate entries to be deleted
    entries_to_delete = []

    for entry_id, entry_data in all_rag_results.items():
        query = entry_data.get('query')
        if query:
            normalized_query = query.lower().strip()
            if normalized_query not in unique_queries:
                unique_queries[normalized_query] = entry_id
            else:
                entries_to_delete.append(entry_id)

    if not entries_to_delete:
        print("No duplicate entries found. Database is clean.")
    else:
        print(f"Found {len(entries_to_delete)} duplicate entries to delete.")
        print("Deleting duplicate entries...")
        for duplicate_id in entries_to_delete:
            try:
                FBconn.delete(f'/rag_search_results/{duplicate_id}', None)
                print(f"  Deleted duplicate entry with ID: {duplicate_id}")
            except Exception as e:
                print(f"  Error deleting {duplicate_id}: {e}")

        print("Duplicate cleaning complete. Verifying...")

        # Verify by fetching again
        verified_results = FBconn.get('/rag_search_results/', None)
        if verified_results:
            print(f"After cleaning, {len(verified_results)} entries remain in Firebase.")
        else:
            print("After cleaning, no entries remain in Firebase.")

        print("Firebase duplicate cleaning script finished.")

Fetching all RAG search results from Firebase...
Found 6 total entries.
Found 2 duplicate entries to delete.
Deleting duplicate entries...
  Deleted duplicate entry with ID: -Otw_lH-1OcMBAo-pozw
  Deleted duplicate entry with ID: -Otw_mUgsVZSHeKCgHM5
Duplicate cleaning complete. Verifying...
After cleaning, 4 entries remain in Firebase.
Firebase duplicate cleaning script finished.


In [12]:
print("Retrieving remaining unique RAG entries from Firebase...")
remaining_results = FBconn.get('/rag_search_results/', None)

if remaining_results:
    print(f"Found {len(remaining_results)} unique entries.")
    for entry_id, entry_data in remaining_results.items():
        query = entry_data.get('query', 'N/A')
        answer = entry_data.get('answer', 'N/A')
        print(f"\n--- Entry ID: {entry_id} ---")
        print(f"  Query: {query}")
        print(f"  Answer: {answer}")
        # print(f"  Context (truncated): {entry_data.get('context', 'N/A')[:200]}...") # Optional: display context
else:
    print("No unique RAG entries found in Firebase after cleaning.")

print("Display of remaining entries finished.")

Retrieving remaining unique RAG entries from Firebase...
Found 4 unique entries.

--- Entry ID: -OtnPjTBxp8IbGIcQpgs ---
  Query: What causes downy mildew in basil and what are the symptoms?
  Answer: Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5].

--- Entry ID: -OtnPjU6U3YvVEXWCCCG ---
  Query: How does temperature affect greenhouse diseases?
  Answer: Based on the provided context, temperature significantly affects greenhouse diseases, specifically in basil crops, in the following ways:

*   **Influences Infection:** Temperature directly affects the infection rate and development of white mould [3]. 
*   **Impacts Spore Inoculum and Sporulation:** For diseases like downy mildew (caused by the oomycete *Peronospora belbahrii*), temperature plays a key role in the disease's epidemiology by driving the development of sp

In [13]:
# Install firebase if not already installed
!pip install firebase PyPDF2 nltk

**RAG**

In [14]:
pass # Obsolete: RAG data setup and functions (non-class based)

# 1. Core RAG Imports and Firebase Connection

In [34]:
import re
import google.generativeai as genai
from firebase.firebase import FirebaseApplication
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer
import datetime # Import datetime module for local timestamps
import PyPDF2 # New import for PDF handling
import tempfile # New import for temporary file handling
import os # New import for file operations

# Initialize Firebase with your database URL
# The 'None' here means no authentication token is provided.
# This setup requires your Firebase Realtime Database security rules to allow unauthenticated write access.
# If you encounter a 401 error, you need to adjust your database rules in the Firebase console.
FBconn = FirebaseApplication('https://basil-plant-disease-default-rtdb.firebaseio.com/', None)

# Configure your API key for the Gemini API
# This key was previously extracted from the kernel state.
API_KEY = ''
genai.configure(api_key=API_KEY)

# 2. RAG Data Setup and Index

In [16]:
pass # Obsolete: Older RAG data setup

# 3. RAG Pipeline Functions

In [17]:
class RAGSystem:
    def __init__(self):
        print("Initializing RAG System...")
        self.doc_id_mapping = {
            1: "https://doi.org/10.3390/plants9050654",
            2: "https://doi.org/10.1094/pdis.1997.81.2.124",
            3: "https://doi.org/10.21273/hortsci09778-16",
            4: "https://doi.org/10.3732/apps.1300032",
            5: "https://doi.org/10.21273/horttech03849-17"
        }

        from nltk.stem import PorterStemmer # Import here to ensure it's available within the class
        self.stemmer = PorterStemmer()

        original_inverted_index = {
            "basil": [1, 2, 3, 5], "disease": [1, 2, 3, 4, 5], "downy": [1, 5],
            "mildew": [1, 5], "management": [1, 2, 3, 5], "infection": [1, 3, 5],
            "resistance": [1, 2, 3, 5], "peronospora": [5], "belbahrii": [5],
            "sporulation": [3, 5], "chlorosis": [5], "oomycete": [5], "spore": [5],
            "greenhouse": [2, 3, 5], "inoculum": [3, 5], "epidemiology": [5],
            "fusarium": [2], "wilt": [2], "mould": [3], "white": [3],
            "epidermis": [5], "temperature": [3, 5]
        }
        self.inverted_index = {}
        for term, doc_ids in original_inverted_index.items():
            self.inverted_index[self.stemmer.stem(term)] = doc_ids

        print("\n--- RAGSystem Stemmed Inverted Index (sample) ---")
        # Print a sample to avoid excessive output, or full for debugging
        sample_keys = list(self.inverted_index.keys())[:5]
        sample_index = {k: self.inverted_index[k] for k in sample_keys}
        print(sample_index)
        print("--------------------------------------------------")

        self.stop_words = {
            "the", "and", "of", "in", "to", "a", "is", "for", "on", "with", "by",
            "from", "at", "as", "it", "he", "she", "they", "we", "you", "that", "this",
            "but", "or", "not", "has", "have", "had", "do", "does", "did", "can",
            "will", "would", "should", "could", "may", "might", "must"
        }

        # Mock corpus to simulate retrieved documents for the RAG pipeline
        self.mock_corpus = {
            1: "Downy mildew is a severe disease in basil. Management requires tracking infection rates and breeding for resistance.",
            2: "Fusarium wilt causes severe disease in greenhouse basil. Proper greenhouse management and disease resistance are key.",
            3: "White mould infection in greenhouse basil is affected by temperature. Managing inoculum and sporulation helps build resistance.",
            4: "General disease management strategies in plants.",
            5: "The oomycete Peronospora belbahrii causes downy mildew in basil. Epidemiology shows temperature impacts spore inoculum and sporulation. Symptoms include chlorosis on the epidermis. Effective disease management and resistance are critical in the greenhouse."
        }
        # Initialize Gemini Model
        self.gemini_model = genai.GenerativeModel('gemini-pro-latest')


    def retrieve_documents(self, query_keywords, top_n=2):
        relevant_doc_ids = set()
        for keyword in query_keywords:
            stemmed_keyword = self.stemmer.stem(keyword.lower())
            if stemmed_keyword in self.inverted_index:
                for doc_id in self.inverted_index[stemmed_keyword]:
                    relevant_doc_ids.add(doc_id)

        sorted_relevant_docs = sorted(list(relevant_doc_ids))
        top_documents = []

        for doc_id in sorted_relevant_docs[:top_n]:
            if doc_id in self.mock_corpus:
                top_documents.append({'id': doc_id, 'text': self.mock_corpus[doc_id]})
        return top_documents

    def generate_rag_response(self, question, context):
        if not context:
            return "i could not find an answer please ask something else"

        context_text = "\n".join([f"Source [{doc['id']}] ({self.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in context])

        prompt = f"""Given the following context, answer the question comprehensively.
        Context:
        {context_text}

        Question: {question}
        Answer:"""

        print("\n--- RAG Prompt (Ready to be sent to LLM) ---")
        print(prompt)
        print("------------------------------------------")

        # Actual LLM integration
        try:
            response = self.gemini_model.generate_content(prompt)
            generated_answer = response.text.strip()
            if not generated_answer:
                return "i could not find an answer to this question according the articles"
            return generated_answer
        except Exception as e:
            return "i could not find an answer please ask something else"

    def get_existing_queries(self, fb_connection):
        try:
            all_rag_results = fb_connection.get('/rag_search_results/', None)
            if all_rag_results:
                return {value.get('query').lower(): key for key, value in all_rag_results.items() if value.get('query')}
            return {}
        except Exception as e:
            print(f"Error fetching existing queries from Firebase: {e}")
            return {}

In [18]:
pass # Obsolete: Duplicates RAG functions

# 4. RAG Pipeline Execution

# 4. RAG Pipeline Execution (using the new `RAGSystem` class)

In [19]:
rag_instance = RAGSystem()

# Fetch existing queries from Firebase
existing_queries = rag_instance.get_existing_queries(FBconn)
print(f"Found {len(existing_queries)} existing queries in Firebase.")

# The LLM is now enabled. No need to configure the API key here again.

# Test Query 1 - Re-processing to find an answer based on articles
test_query = "What causes downy mildew in basil and what are the symptoms?"
print(f"\nRe-processing Query 1: {test_query}")

# Explicitly construct context from relevant documents in mock_corpus
# Documents 1 and 5 directly mention downy mildew, basil, and symptoms
relevant_docs_for_mildew = [
    {'id': 1, 'text': rag_instance.mock_corpus[1]},
    {'id': 5, 'text': rag_instance.mock_corpus[5]}
]

# Ensure retrieved_context uses these specific documents
retrieved_context = relevant_docs_for_mildew
context_q1 = "\n".join([f"Source [{doc['id']}] ({rag_instance.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in retrieved_context])

print(f"Explicitly constructed Context for Query 1:\n{context_q1}")

# This will now use the actual LLM to generate a new answer
answer = rag_instance.generate_rag_response(test_query, retrieved_context)
print(f"Generated Answer 1: {answer}")

# Get the existing entry ID for this query from the previous output
existing_entry_id_q1 = '-OtnPjTBxp8IbGIcQpgs' # This ID was confirmed in the `remaining_results` output

# Update the answer and context for this specific entry in Firebase
try:
    rag_data_update = {
        'answer': answer,
        'context': context_q1,
        'timestamp_updated': datetime.datetime.now().isoformat() # Add an update timestamp
    }
    FBconn.patch(f'/rag_search_results/{existing_entry_id_q1}/', rag_data_update)
    print(f"Successfully updated Query 1 data for ID {existing_entry_id_q1} in Firebase with new answer.")
except Exception as e:
    print(f"Error updating Query 1 data for ID {existing_entry_id_q1} in Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")

# Test Query 2 - Remains as is (already processed with an answer)
test_query2 = "How does temperature affect greenhouse diseases?"
print(f"\nQuery 2: {test_query2} (Already processed in previous turn)")
existing_entry_id_q2 = '-OtnPjU6U3YvVEXWCCCG'
existing_entry_q2 = FBconn.get(f'/rag_search_results/{existing_entry_id_q2}', None)
if existing_entry_q2:
    print(f"Existing Answer: {existing_entry_q2.get('answer', 'N/A')}")
else:
    print("Could not retrieve existing answer for Query 2.")

Initializing RAG System...

--- RAGSystem Stemmed Inverted Index (sample) ---
{'basil': [1, 2, 3, 5], 'diseas': [1, 2, 3, 4, 5], 'downi': [1, 5], 'mildew': [1, 5], 'manag': [1, 2, 3, 5]}
--------------------------------------------------
Found 4 existing queries in Firebase.

Re-processing Query 1: What causes downy mildew in basil and what are the symptoms?
Explicitly constructed Context for Query 1:
Source [1] (https://doi.org/10.3390/plants9050654): Downy mildew is a severe disease in basil. Management requires tracking infection rates and breeding for resistance.
Source [5] (https://doi.org/10.21273/horttech03849-17): The oomycete Peronospora belbahrii causes downy mildew in basil. Epidemiology shows temperature impacts spore inoculum and sporulation. Symptoms include chlorosis on the epidermis. Effective disease management and resistance are critical in the greenhouse.

--- RAG Prompt (Ready to be sent to LLM) ---
Given the following context, answer the question comprehensively.
 

### Store RAG Configuration and Knowledge Base to Firebase

This cell will explicitly store the `doc_id_mapping`, `inverted_index`, and `mock_corpus` of the `RAGSystem` instance to Firebase. This helps in auditing the RAG system's knowledge base and configuration at different points in time.

In [20]:
# Assuming 'rag_instance' is already initialized from the previous cell

try:
    # Store doc_id_mapping
    FBconn.put('/rag_config/', 'doc_id_mapping', rag_instance.doc_id_mapping)
    print("Successfully posted doc_id_mapping to Firebase.")

    # Store inverted_index (could be large, consider breaking it down if needed)
    FBconn.put('/rag_config/', 'inverted_index', rag_instance.inverted_index)
    print("Successfully posted inverted_index to Firebase.")

    # Store mock_corpus
    FBconn.put('/rag_config/', 'mock_corpus', rag_instance.mock_corpus)
    print("Successfully posted mock_corpus to Firebase.")

    # Optionally, store a timestamp for this configuration update
    FBconn.post('/rag_config_updates/', {'timestamp': datetime.datetime.now().isoformat(), 'message': 'RAG system configuration updated'})
    print("Configuration update timestamp posted to Firebase.")

except Exception as e:
    print(f"Error posting RAG configuration to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_config/' and '/rag_config_updates/' to ensure write access is permitted.")

Successfully posted doc_id_mapping to Firebase.
Successfully posted inverted_index to Firebase.
Successfully posted mock_corpus to Firebase.
Configuration update timestamp posted to Firebase.


### Re-storing RAG Configuration and Knowledge Base to Firebase

This cell explicitly re-stores the `doc_id_mapping`, `inverted_index`, and `mock_corpus` of the `RAGSystem` instance to Firebase after the class modification. This ensures the database reflects the latest RAG system's knowledge base and configuration.

In [21]:
# Assuming 'rag_instance' is already initialized and updated from the previous cells

try:
    # Store doc_id_mapping
    FBconn.put('/rag_config/', 'doc_id_mapping', rag_instance.doc_id_mapping)
    print("Successfully posted doc_id_mapping to Firebase.")

    # Store inverted_index (could be large, consider breaking it down if needed)
    FBconn.put('/rag_config/', 'inverted_index', rag_instance.inverted_index)
    print("Successfully posted inverted_index to Firebase.")

    # Store mock_corpus
    FBconn.put('/rag_config/', 'mock_corpus', rag_instance.mock_corpus)
    print("Successfully posted mock_corpus to Firebase.")

    # Optionally, store a timestamp for this configuration update
    FBconn.post('/rag_config_updates/', {'timestamp': datetime.datetime.now().isoformat(), 'message': 'RAG system configuration re-updated'})
    print("Configuration re-update timestamp posted to Firebase.")

except Exception as e:
    print(f"Error posting RAG configuration to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_config/' and '/rag_config_updates/' to ensure write access is permitted.")

Successfully posted doc_id_mapping to Firebase.
Successfully posted inverted_index to Firebase.
Successfully posted mock_corpus to Firebase.
Configuration re-update timestamp posted to Firebase.


### Explicitly Updating a Specific RAG Result in Firebase

This cell updates the `answer` field for a specific RAG search result entry in Firebase, as per your request.

In [22]:
specific_entry_id = '-OtnPjTBxp8IbGIcQpgs'
new_answer_text = 'i could not find an answer to this question according the articles'

try:
    # Update only the 'answer' field of the specific entry
    FBconn.patch(f'/rag_search_results/{specific_entry_id}/', {'answer': new_answer_text})
    print(f"Successfully updated answer for entry ID {specific_entry_id} in Firebase.")

except Exception as e:
    print(f"Error updating entry ID {specific_entry_id} in Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")

Successfully updated answer for entry ID -OtnPjTBxp8IbGIcQpgs in Firebase.


### Verifying the Specific Updated RAG Result in Firebase

This cell retrieves and displays the specific RAG search result entry that was just updated, allowing you to confirm the change.

In [23]:
specific_entry_id = '-OtnPjTBxp8IbGIcQpgs'

try:
    # Get the specific RAG search result
    updated_entry = FBconn.get(f'/rag_search_results/{specific_entry_id}', None)

    if updated_entry:
        print(f"Successfully retrieved updated RAG search result for ID: {specific_entry_id}")
        print("\n--- Updated RAG Search Result ---")
        print(f"  Query: {updated_entry.get('query', 'N/A')}")
        print(f"  Answer: {updated_entry.get('answer', 'N/A')}")
        print(f"  Context: {updated_entry.get('context', 'N/A')[:200]}...") # Truncate long context
        print(f"  Timestamp: {updated_entry.get('timestamp', 'N/A')}")
        print("--------------------------------------------------")
    else:
        print(f"No RAG search result found for ID: {specific_entry_id}.")

except Exception as e:
    print(f"Error retrieving RAG search result for ID {specific_entry_id} from Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure read access is permitted.")

Successfully retrieved updated RAG search result for ID: -OtnPjTBxp8IbGIcQpgs

--- Updated RAG Search Result ---
  Query: What causes downy mildew in basil and what are the symptoms?
  Answer: i could not find an answer to this question according the articles
  Context: Source [1] (https://doi.org/10.3390/plants9050654): Downy mildew is a severe disease in basil. Management requires tracking infection rates and breeding for resistance.
Source [5] (https://doi.org/10....
  Timestamp: 2026-05-29T11:07:30.525740
--------------------------------------------------


### Updating a Specific RAG Result in Firebase with a Context-Derived Answer

This cell explicitly updates the `answer` field for the specific RAG search result entry in Firebase (`-OtnPjTBxp8IbGIcQpgs`) with an answer derived directly from the provided context.

In [24]:
specific_entry_id = '-OtnPjTBxp8IbGIcQpgs'
new_context_derived_answer = "Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5]."

try:
    # Update only the 'answer' field of the specific entry
    FBconn.patch(f'/rag_search_results/{specific_entry_id}/', {'answer': new_context_derived_answer, 'timestamp_updated': datetime.datetime.now().isoformat()})
    print(f"Successfully updated answer for entry ID {specific_entry_id} in Firebase with context-derived answer.")

except Exception as e:
    print(f"Error updating entry ID {specific_entry_id} in Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")

Successfully updated answer for entry ID -OtnPjTBxp8IbGIcQpgs in Firebase with context-derived answer.


### Verifying the New Context-Derived Answer

This cell retrieves and displays the specific RAG search result entry that was just updated, allowing you to confirm the change to the context-derived answer.

In [25]:
specific_entry_id = '-OtnPjTBxp8IbGIcQpgs'

try:
    # Get the specific RAG search result
    updated_entry = FBconn.get(f'/rag_search_results/{specific_entry_id}', None)

    if updated_entry:
        print(f"Successfully retrieved updated RAG search result for ID: {specific_entry_id}")
        print("\n--- Updated RAG Search Result ---")
        print(f"  Query: {updated_entry.get('query', 'N/A')}")
        print(f"  Answer: {updated_entry.get('answer', 'N/A')}")
        print(f"  Context: {updated_entry.get('context', 'N/A')[:200]}...") # Truncate long context
        print(f"  Timestamp: {updated_entry.get('timestamp', 'N/A')}")
        print(f"  Timestamp Updated: {updated_entry.get('timestamp_updated', 'N/A')}")
        print("--------------------------------------------------")
    else:
        print(f"No RAG search result found for ID: {specific_entry_id}.")

except Exception as e:
    print(f"Error retrieving RAG search result for ID {specific_entry_id} from Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure read access is permitted.")

Successfully retrieved updated RAG search result for ID: -OtnPjTBxp8IbGIcQpgs

--- Updated RAG Search Result ---
  Query: What causes downy mildew in basil and what are the symptoms?
  Answer: Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5].
  Context: Source [1] (https://doi.org/10.3390/plants9050654): Downy mildew is a severe disease in basil. Management requires tracking infection rates and breeding for resistance.
Source [5] (https://doi.org/10....
  Timestamp: 2026-05-29T11:07:30.525740
  Timestamp Updated: 2026-05-31T05:52:34.713245
--------------------------------------------------


### Adding Two New Questions and Their Answers to Firebase

This section adds two new RAG queries along with their context-derived answers to the Firebase database. Each query is processed using the `RAGSystem` to ensure answers are based on the `mock_corpus`.

In [26]:
# --- New Query 1: What causes fusarium wilt in basil? ---
print("\n--- Processing New Query 1 ---")
new_query_1 = "What causes fusarium wilt in basil?"

# Manually select relevant documents for this query from mock_corpus
# Article 2 is directly relevant
relevant_docs_q1 = [
    {'id': 2, 'text': rag_instance.mock_corpus[2]}
]
context_new_q1 = "\n".join([f"Source [{doc['id']}] ({rag_instance.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in relevant_docs_q1])

print(f"New Query 1: {new_query_1}")
print(f"Context for New Query 1:\n{context_new_q1}")

answer_new_q1 = rag_instance.generate_rag_response(new_query_1, relevant_docs_q1)
print(f"Generated Answer for New Query 1: {answer_new_q1}")

# Post the new query and answer to Firebase
try:
    rag_data_new_q1 = {
        'query': new_query_1,
        'answer': answer_new_q1,
        'context': context_new_q1,
        'timestamp': datetime.datetime.now().isoformat()
    }
    post_result_q1 = FBconn.post('/rag_search_results/', rag_data_new_q1)
    print(f"Successfully posted New Query 1 to Firebase with ID: {post_result_q1['name']}")
except Exception as e:
    print(f"Error posting New Query 1 to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")


# --- New Query 2: What affects white mould infection in greenhouse basil? ---
print("\n--- Processing New Query 2 ---")
new_query_2 = "What affects white mould infection in greenhouse basil?"

# Manually select relevant documents for this query from mock_corpus
# Article 3 is directly relevant
relevant_docs_q2 = [
    {'id': 3, 'text': rag_instance.mock_corpus[3]}
]
context_new_q2 = "\n".join([f"Source [{doc['id']}] ({rag_instance.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in relevant_docs_q2])

print(f"New Query 2: {new_query_2}")
print(f"Context for New Query 2:\n{context_new_q2}")

answer_new_q2 = rag_instance.generate_rag_response(new_query_2, relevant_docs_q2)
print(f"Generated Answer for New Query 2: {answer_new_q2}")

# Post the new query and answer to Firebase
try:
    rag_data_new_q2 = {
        'query': new_query_2,
        'answer': answer_new_q2,
        'context': context_new_q2,
        'timestamp': datetime.datetime.now().isoformat()
    }
    post_result_q2 = FBconn.post('/rag_search_results/', rag_data_new_q2)
    print(f"Successfully posted New Query 2 to Firebase with ID: {post_result_q2['name']}")
except Exception as e:
    print(f"Error posting New Query 2 to Firebase: {e}")
    print("Please check your Firebase database rules for '/rag_search_results/' to ensure write access is permitted.")


--- Processing New Query 1 ---
New Query 1: What causes fusarium wilt in basil?
Context for New Query 1:
Source [2] (https://doi.org/10.1094/pdis.1997.81.2.124): Fusarium wilt causes severe disease in greenhouse basil. Proper greenhouse management and disease resistance are key.

--- RAG Prompt (Ready to be sent to LLM) ---
Given the following context, answer the question comprehensively.
        Context:
        Source [2] (https://doi.org/10.1094/pdis.1997.81.2.124): Fusarium wilt causes severe disease in greenhouse basil. Proper greenhouse management and disease resistance are key.

        Question: What causes fusarium wilt in basil?
        Answer:
------------------------------------------
Generated Answer for New Query 1: Based on the provided context, the exact underlying cause (such as the specific pathogen) of Fusarium wilt is not explicitly stated. Instead, the context explains that Fusarium wilt itself *is* the cause of a severe disease affecting greenhouse basil. 

To ad

### Displaying All RAG Entries After Additions

This cell retrieves and displays all RAG queries and their answers that are currently stored in your Firebase database, including the newly added questions.

In [27]:
print("Retrieving all RAG entries from Firebase to confirm additions...")
all_current_results = FBconn.get('/rag_search_results/', None)

if all_current_results:
    print(f"Found {len(all_current_results)} total entries.")
    for entry_id, entry_data in all_current_results.items():
        query = entry_data.get('query', 'N/A')
        answer = entry_data.get('answer', 'N/A')
        timestamp = entry_data.get('timestamp', 'N/A')
        timestamp_updated = entry_data.get('timestamp_updated', 'N/A')
        print(f"\n--- Entry ID: {entry_id} ---")
        print(f"  Query: {query}")
        print(f"  Answer: {answer}")
        print(f"  Timestamp: {timestamp}")
        if timestamp_updated != 'N/A':
            print(f"  Timestamp Updated: {timestamp_updated}")
else:
    print("No RAG entries found in Firebase.")

print("Display of all current entries finished.")

Retrieving all RAG entries from Firebase to confirm additions...
Found 6 total entries.

--- Entry ID: -OtnPjTBxp8IbGIcQpgs ---
  Query: What causes downy mildew in basil and what are the symptoms?
  Answer: Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5].
  Timestamp: 2026-05-29T11:07:30.525740
  Timestamp Updated: 2026-05-31T05:52:34.713245

--- Entry ID: -OtnPjU6U3YvVEXWCCCG ---
  Query: How does temperature affect greenhouse diseases?
  Answer: Based on the provided context, temperature significantly affects greenhouse diseases, specifically in basil crops, in the following ways:

*   **Influences Infection:** Temperature directly affects the infection rate and development of white mould [3]. 
*   **Impacts Spore Inoculum and Sporulation:** For diseases like downy mildew (caused by the oomycete *Peronospora belbahrii

In [28]:
pass # Obsolete: Older RAG pipeline execution

### Cleaning Duplicate RAG Entries from Firebase

This script will identify and remove duplicate RAG query entries from your Firebase database. It will fetch all entries under `/rag_search_results/`, group them by the 'query' field, and then delete all but the first occurrence of each unique query.


In [29]:
# Fetch all RAG search results
print("Fetching all RAG search results from Firebase...")
all_rag_results = FBconn.get('/rag_search_results/', None)

if not all_rag_results:
    print("No RAG search results found in Firebase.")
else:
    print(f"Found {len(all_rag_results)} total entries.")

    # Dictionary to store unique queries and their first entry ID
    unique_queries = {}
    # List to store IDs of duplicate entries to be deleted
    entries_to_delete = []

    for entry_id, entry_data in all_rag_results.items():
        query = entry_data.get('query')
        if query:
            normalized_query = query.lower().strip()
            if normalized_query not in unique_queries:
                unique_queries[normalized_query] = entry_id
            else:
                entries_to_delete.append(entry_id)

    if not entries_to_delete:
        print("No duplicate entries found. Database is clean.")
    else:
        print(f"Found {len(entries_to_delete)} duplicate entries to delete.")
        print("Deleting duplicate entries...")
        for duplicate_id in entries_to_delete:
            try:
                FBconn.delete(f'/rag_search_results/{duplicate_id}', None)
                print(f"  Deleted duplicate entry with ID: {duplicate_id}")
            except Exception as e:
                print(f"  Error deleting {duplicate_id}: {e}")

        print("Duplicate cleaning complete. Verifying...")

        # Verify by fetching again
        verified_results = FBconn.get('/rag_search_results/', None)
        if verified_results:
            print(f"After cleaning, {len(verified_results)} entries remain in Firebase.")
        else:
            print("After cleaning, no entries remain in Firebase.")

        print("Firebase duplicate cleaning script finished.")


Fetching all RAG search results from Firebase...
Found 6 total entries.
Found 2 duplicate entries to delete.
Deleting duplicate entries...
  Deleted duplicate entry with ID: -Otw_rpS-P4OKF1mFb7L
  Deleted duplicate entry with ID: -Otw_ssavUF1ei7d8-vk
Duplicate cleaning complete. Verifying...
After cleaning, 4 entries remain in Firebase.
Firebase duplicate cleaning script finished.


### Remaining Unique RAG Entries in Firebase

After the duplicate cleaning, let's retrieve and display the unique RAG queries and their answers that are currently stored in your Firebase database.

In [30]:
print("Retrieving remaining unique RAG entries from Firebase...")
remaining_results = FBconn.get('/rag_search_results/', None)

if remaining_results:
    print(f"Found {len(remaining_results)} unique entries.")
    for entry_id, entry_data in remaining_results.items():
        query = entry_data.get('query', 'N/A')
        answer = entry_data.get('answer', 'N/A')
        print(f"\n--- Entry ID: {entry_id} ---")
        print(f"  Query: {query}")
        print(f"  Answer: {answer}")
        # print(f"  Context (truncated): {entry_data.get('context', 'N/A')[:200]}...") # Optional: display context
else:
    print("No unique RAG entries found in Firebase after cleaning.")

print("Display of remaining entries finished.")

Retrieving remaining unique RAG entries from Firebase...
Found 4 unique entries.

--- Entry ID: -OtnPjTBxp8IbGIcQpgs ---
  Query: What causes downy mildew in basil and what are the symptoms?
  Answer: Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahrii* [5]. The symptom of the disease mentioned in the text is chlorosis (yellowing) on the epidermis of the plant [5].

--- Entry ID: -OtnPjU6U3YvVEXWCCCG ---
  Query: How does temperature affect greenhouse diseases?
  Answer: Based on the provided context, temperature significantly affects greenhouse diseases, specifically in basil crops, in the following ways:

*   **Influences Infection:** Temperature directly affects the infection rate and development of white mould [3]. 
*   **Impacts Spore Inoculum and Sporulation:** For diseases like downy mildew (caused by the oomycete *Peronospora belbahrii*), temperature plays a key role in the disease's epidemiology by driving the development of sp

In [31]:
pass # Obsolete: Model listing cell

In [32]:
pass # Obsolete: Model listing cell